In [2]:
import numpy as np
import pandas as pd

In [3]:
import numpy as np
import pandas as pd

In [4]:
np.random.seed(42)

In [5]:
history = pd.DataFrame({

    'Amount': np.random.normal(
        1500, 400, 1000
    ),

    'Transactions_Per_Hour': np.random.normal(
        2, 0.7, 1000
    )

})

In [6]:
history['Amount'] = history['Amount'].clip(lower=100)

history['Transactions_Per_Hour'] = (
    history['Transactions_Per_Hour'].clip(lower=0.1)
)

print(history.head())
print(history.shape)

        Amount  Transactions_Per_Hour
0  1698.685661               2.979549
1  1444.694280               2.647244
2  1759.075415               2.041741
3  2109.211943               1.547144
4  1406.338650               2.488756
(1000, 2)


In [7]:
new_transactions = pd.DataFrame({

    'Amount': [
        900, 1800, 2500, 95000,
        75000, 200, 45000, 1300,
        2000, 3000, 600, 1800
    ],

    'Transactions_Per_Hour': [
        1, 2, 2, 12,
        9, 15, 1, 1.5,
        4, 3, 2, 1
    ]

})

print(new_transactions)

    Amount  Transactions_Per_Hour
0      900                    1.0
1     1800                    2.0
2     2500                    2.0
3    95000                   12.0
4    75000                    9.0
5      200                   15.0
6    45000                    1.0
7     1300                    1.5
8     2000                    4.0
9     3000                    3.0
10     600                    2.0
11    1800                    1.0


In [8]:
from sklearn.preprocessing import StandardScaler

features = [
    'Amount',
    'Transactions_Per_Hour'
]

scaler = StandardScaler()

X_train = scaler.fit_transform(
    history[features]
)

X_new = scaler.transform(
    new_transactions[features]
)

In [9]:
#train all 4 algorithms
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope

models = {

    'Isolation Forest': IsolationForest(
        n_estimators=200,
        contamination=0.02,
        random_state=42
    ),

    'Local Outlier Factor': LocalOutlierFactor(
        n_neighbors=20,
        contamination=0.02,
        novelty=True
    ),

    'One Class SVM': OneClassSVM(
        kernel='rbf',
        gamma='scale',
        nu=0.02
    ),

    'Elliptic Envelope': EllipticEnvelope(
        contamination=0.02,
        random_state=42
    )

}

In [10]:
#train & predict
results = new_transactions.copy()

for name, model in models.items():

    # Train using historical data
    model.fit(X_train)

    # Predict incoming transactions
    predictions = model.predict(X_new)

    # Convert predictions into readable labels
    results[name] = [
        'Anomaly' if value == -1 else 'Normal'
        for value in predictions
    ]

print(results.to_string(index=False))

 Amount  Transactions_Per_Hour Isolation Forest Local Outlier Factor One Class SVM Elliptic Envelope
    900                    1.0           Normal               Normal        Normal            Normal
   1800                    2.0           Normal               Normal        Normal            Normal
   2500                    2.0           Normal               Normal        Normal            Normal
  95000                   12.0          Anomaly              Anomaly       Anomaly           Anomaly
  75000                    9.0          Anomaly              Anomaly       Anomaly           Anomaly
    200                   15.0          Anomaly              Anomaly       Anomaly           Anomaly
  45000                    1.0          Anomaly              Anomaly       Anomaly           Anomaly
   1300                    1.5           Normal               Normal        Normal            Normal
   2000                    4.0          Anomaly              Anomaly       Anomaly         